## Transcripción de audio en Whisper

In [9]:
# Instalamos la librería de Whisper desde el repositorio oficial
%pip install git+https://github.com/openai/whisper.git

# Aseguramos que tenemos ffmpeg (necesario para procesar audio)
%load_ext autoreload
%autoreload 2
%pip install imageio-ffmpeg

  Cloning https://github.com/openai/whisper.git to /tmp/pip-req-build-oehr2fcy
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /tmp/pip-req-build-oehr2fcy
  Resolved https://github.com/openai/whisper.git to commit c0d2f624c09dc18e709e37c2ad90c039a4eb72a2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Implantando FFMPEG

In [14]:
import os
import stat
import shutil
from pathlib import Path
import imageio_ffmpeg

# Ruta real del binario descargado por imageio-ffmpeg
src = Path(imageio_ffmpeg.get_ffmpeg_exe())
print("Binario real:", src)

# Carpeta de usuario sin permisos de admin
user_bin = Path.home() / ".local" / "bin"
user_bin.mkdir(parents=True, exist_ok=True)

# Crear un enlace llamado exactamente "ffmpeg"
dst = user_bin / "ffmpeg"

if dst.exists() or dst.is_symlink():
    dst.unlink()

dst.symlink_to(src)

# Asegurar permisos de ejecución por si acaso
dst.chmod(dst.stat().st_mode | stat.S_IEXEC)

# Añadir al PATH para esta sesión
os.environ["PATH"] = str(user_bin) + os.pathsep + os.environ.get("PATH", "")

print("Enlace creado:", dst)
print("ffmpeg en PATH:", shutil.which("ffmpeg"))


Binario real: /home/ciabd10/anaconda3/lib/python3.13/site-packages/imageio_ffmpeg/binaries/ffmpeg-linux-x86_64-v7.0.2
Enlace creado: /home/ciabd10/.local/bin/ffmpeg
ffmpeg en PATH: /home/ciabd10/.local/bin/ffmpeg


In [15]:
import subprocess
subprocess.run(["ffmpeg", "-version"], check=True)


ffmpeg version 7.0.2-static https://johnvansickle.com/ffmpeg/  Copyright (c) 2000-2024 the FFmpeg developers
built with gcc 8 (Debian 8.3.0-6)
configuration: --enable-gpl --enable-version3 --enable-static --disable-debug --disable-ffplay --disable-indev=sndio --disable-outdev=sndio --cc=gcc --enable-fontconfig --enable-frei0r --enable-gnutls --enable-gmp --enable-libgme --enable-gray --enable-libaom --enable-libfribidi --enable-libass --enable-libvmaf --enable-libfreetype --enable-libmp3lame --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-librubberband --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libvorbis --enable-libopus --enable-libtheora --enable-libvidstab --enable-libvo-amrwbenc --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libdav1d --enable-libxvid --enable-libzvbi --enable-libzimg
libavutil      59.  8.100 / 59.  8.100
libavcodec     61.  3.100 / 61.  3.100
libavformat    61.  1.10

CompletedProcess(args=['ffmpeg', '-version'], returncode=0)

#### Cargando modelo de Whisper

In [11]:
import whisper
import os

# Elegimos el modelo (puedes cambiar "base" por "small" o "medium" según tu PC/GPU)
model = whisper.load_model("base")
print("Modelo cargado correctamente.")

Modelo cargado correctamente.


#### Cargar el archivo de audio

In [12]:
audio_path = "prueba_larga_whisper.mp3" # Cambia esto por el nombre de tu archivo

if not os.path.exists(audio_path):
    print(f"Error: El archivo {audio_path} no existe. Por favor, súbelo.")
else:
    print(f"Archivo {audio_path} listo para procesar.")

Archivo prueba_larga_whisper.mp3 listo para procesar.


#### Realizando la transcripcion

In [17]:
# Transcribir el audio
result = model.transcribe(audio_path, verbose=False)

# El resultado es un diccionario que contiene el texto y otros metadatos
print("-" * 30)
print("TRANSCRIPCIÓN FINALIZADA")
print("-" * 30)
print(result["text"])

# Guardando la transcripción en un archivo de texto
nombre_salida = "transcripcion_resultado.txt"
with open(nombre_salida, "w", encoding="utf-8") as f:
    f.write(result["text"])
print(f"La transcripción se ha guardado en: {nombre_salida}")


Detected language: Spanish


100%|██████████| 4691/4691 [00:00<00:00, 6089.98frames/s]

------------------------------
TRANSCRIPCIÓN FINALIZADA
------------------------------
 Esta es una nueva laudión poco más larga para verificar el funciónamiento del whisper. Estoy hablando en español con una velocidad normal y con frases sencillas para que el modelo pueda reconocer bien las palabras. Este aquí lo sirve para controlar la transcripción, la puntuación y también el comportamiento general del sistema con una muestra de voz continua. Si todo funciona correctamente, el resultado debe y aparecerse bastante al texto original, aunque puede haber pequeñas diferencias en algunas palabras en los signos de puntuación. Esta prueba también puede ayudar a comparar distintos modelos, con figuras y niveles de compresión de audio.
La transcripción se ha guardado en: transcripcion_resultado.txt


#### Transcripción con marcas de tiempo

In [18]:
for segment in result['segments']:
    start = segment['start']
    end = segment['end']
    text = segment['text']
    print(f"[{start:5.2f}s -> {end:5.2f}s] {text}")

[ 0.00s ->  7.00s]  Esta es una nueva laudión poco más larga para verificar el funciónamiento del whisper.
[ 7.00s -> 13.00s]  Estoy hablando en español con una velocidad normal y con frases sencillas para que el modelo
[13.00s -> 19.00s]  pueda reconocer bien las palabras. Este aquí lo sirve para controlar la transcripción,
[19.00s -> 25.00s]  la puntuación y también el comportamiento general del sistema con una muestra de voz continua.
[25.00s -> 32.00s]  Si todo funciona correctamente, el resultado debe y aparecerse bastante al texto original,
[32.00s -> 38.00s]  aunque puede haber pequeñas diferencias en algunas palabras en los signos de puntuación.
[38.00s -> 46.00s]  Esta prueba también puede ayudar a comparar distintos modelos, con figuras y niveles de compresión de audio.


#### Traducir al inglés

In [19]:
# Usamos el parámetro task="translate"
result_en = model.transcribe(audio_path, task="translate")
print(result_en["text"])

 This is a new audio a little longer to verify the function of the whisper. I am talking in Spanish with a normal and common phrase in simple so that the model can recognize the words. This file can be checked by transcription, the punctuation and also the general behavior of the system that this new neighbor continues. If the result is correct, the result should not be enough to the original, although there may be small differences in some words or in the punctuation signals. This may also help to compare the different models, configurations or levels of compression of audio.
